In [1]:
# Import necessary modules
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from database import db
from attendance import attendance_manager
from excel_report import excel_report

2026-07-03 11:28:12,603 - database - INFO - Successfully connected to MongoDB database: attendance_system
2026-07-03 11:28:21,314 - keras_facenet.embedding_model - INFO - Loading weights.
2026-07-03 11:28:21,318 - keras_facenet.utils - INFO - Looking for C:\Users\sanja/.keras-facenet\20180402-114759\20180402-114759-weights.h5
2026-07-03 11:28:27,467 - tensorflow - WARNING - From C:\Users\sanja\OneDrive\Desktop\attendance_system\venv\Lib\site-packages\keras\src\backend\tensorflow\core.py:233: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

2026-07-03 11:28:28,767 - embedding_generator - INFO - FaceNet model loaded successfully
2026-07-03 11:28:29,199 - recognition - WARNING - MediaPipe initialization failed: module 'mediapipe.tasks.python.vision.face_detector' has no attribute 'RunningMode'
2026-07-03 11:28:29,200 - recognition - INFO - Falling back to OpenCV Haar Cascade...
2026-07-03 11:28:29,244 - recognition - INFO - Using OpenCV Haar Cascade fro

In [2]:
# List available attendance sessions
collection = db.get_collection('attendance')

# Get unique session IDs
sessions = collection.distinct('session_id')

print(f"Available Attendance Sessions: {len(sessions)}")
print("-" * 40)

for session_id in sorted(sessions)[-10:]:  # Show last 10 sessions
    session_records = attendance_manager.get_attendance_by_session(session_id)

    if session_records:
        first_record = session_records[0]

        present = sum(
            1 for r in session_records
            if r.get('status') == 'present'
        )

        total = len(session_records)

        print(
            f"{session_id} | "
            f"{first_record.get('date', 'N/A')} | "
            f"{first_record.get('subject', 'N/A')} | "
            f"{present}/{total} present"
        )

Available Attendance Sessions: 2
----------------------------------------
ATT_20260703_112236_cc77aa | 2026-07-03 | DS | 1/5 present
ATT_20260703_112655_d25f21 | 2026-07-03 | ML | 1/5 present


In [3]:
# Generate report for a specific session
session_id = input("\nEnter session ID to generate report: ").strip()

if session_id:

    try:

        report_path = excel_report.generate_attendance_report(session_id)

        print("\n✓ Report generated successfully")
        print(f"Location : {report_path}")
        print(f"File Size: {report_path.stat().st_size / 1024:.2f} KB")

        # Preview the report
        df = pd.read_excel(report_path, header=7)

        print("\nReport Preview")
        print("-" * 60)

        print(df.head(10).to_string())

    except Exception as e:

        print(f"\n✗ Error generating report: {e}")


Enter session ID to generate report:  ATT_20260703_112655_d25f21 


2026-07-03 11:29:06,726 - excel_report - INFO - Attendance report generated: C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\2026-07-03\Attendance_ATT_20260703_112655_d25f21_2026-07-03.xlsx



✓ Report generated successfully
Location : C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\2026-07-03\Attendance_ATT_20260703_112655_d25f21_2026-07-03.xlsx
File Size: 5.73 KB

Report Preview
------------------------------------------------------------
        Date:     2026-07-03        Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5  Unnamed: 6
0       Time:       11:27:33               NaN        NaN        NaN        NaN         NaN
1  Classroom:         SR 306               NaN        NaN        NaN        NaN         NaN
2         NaN            NaN               NaN        NaN        NaN        NaN         NaN
3     Roll No   Student Name        Department    Section     Status       Time  Confidence
4       22001    Rahul Kumar  Computer Science          A    PRESENT   11:27:33       68.4%
5       22002   Priya Sharma  Computer Science          A     ABSENT         --          --
6       22003      Sai Reddy  Computer Science          A     ABSENT         --          -

In [4]:
# Generate department report

departments = [
    "Computer Science",
    "Electronics",
    "Mechanical",
    "Civil"
]

print("Available Departments")

for i, dept in enumerate(departments, 1):
    print(f"{i}. {dept}")

choice = input("\nEnter department number or department name: ").strip()

try:

    index = int(choice) - 1

    if 0 <= index < len(departments):
        department = departments[index]
    else:
        department = choice

except:

    department = choice

if department:

    try:

        date = input(
            "Enter date (YYYY-MM-DD) or press Enter for today: "
        ).strip()

        if not date:
            date = datetime.now().strftime("%Y-%m-%d")

        report_path = excel_report.generate_department_report(
            department,
            date
        )

        print(f"\n✓ Department Report Generated")
        print(report_path)

    except Exception as e:

        print(f"\n✗ Error: {e}")

Available Departments
1. Computer Science
2. Electronics
3. Mechanical
4. Civil



Enter department number or department name:  1
Enter date (YYYY-MM-DD) or press Enter for today:  


2026-07-03 11:29:21,522 - excel_report - INFO - Department report generated: C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\department_reports\Computer Science_Attendance_2026-07-03.xlsx



✓ Department Report Generated
C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\department_reports\Computer Science_Attendance_2026-07-03.xlsx


In [5]:
# Generate student report

roll_number = input("\nEnter student roll number: ").strip()

if roll_number:

    try:

        report_path = excel_report.generate_student_report(
            roll_number
        )

        print("\n✓ Student Report Generated")
        print(report_path)

    except Exception as e:

        print(f"\n✗ Error: {e}")


Enter student roll number:  22001


2026-07-03 11:29:25,599 - excel_report - INFO - Student report generated: C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\student_reports\22001_Rahul Kumar_Attendance_2026-07-03.xlsx



✓ Student Report Generated
C:\Users\sanja\OneDrive\Desktop\attendance_system\Attendance\student_reports\22001_Rahul Kumar_Attendance_2026-07-03.xlsx


In [6]:
from datetime import timedelta

print("\nGenerate Consolidated Attendance Report")
print("-" * 50)

start_date = input(
    "Start Date (YYYY-MM-DD) or press Enter for last 7 days: "
).strip()

end_date = input(
    "End Date (YYYY-MM-DD) or press Enter for today: "
).strip()

if not start_date:
    start_date = (
        datetime.now() - timedelta(days=7)
    ).strftime("%Y-%m-%d")

if not end_date:
    end_date = datetime.now().strftime("%Y-%m-%d")

collection = db.get_collection("attendance")

query = {
    "date": {
        "$gte": start_date,
        "$lte": end_date
    }
}

records = list(collection.find(query))

if records:

    print(f"\nFound {len(records)} attendance records")

    df = pd.DataFrame(records)

    columns = [
        "roll_number",
        "name",
        "department",
        "section",
        "subject",
        "status",
        "date",
        "time_in"
    ]

    available_columns = [
        col for col in columns if col in df.columns
    ]

    df = df[available_columns]

    filename = (
        f"Consolidated_Attendance_{start_date}_to_{end_date}.xlsx"
    )

    filepath = Path("../Attendance") / filename

    with pd.ExcelWriter(
        filepath,
        engine="openpyxl"
    ) as writer:

        df.to_excel(
            writer,
            sheet_name="Attendance",
            index=False
        )

        summary = (
            df.groupby(["roll_number", "name"])
            .agg({
                "status": lambda x:
                    f"{sum(x == 'present')}/{len(x)}"
            })
            .reset_index()
        )

        summary.columns = [
            "Roll Number",
            "Name",
            "Attendance"
        ]

        summary.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

    print(f"\n✓ Consolidated Report Generated")
    print(filepath)

else:

    print("No attendance records found.")


Generate Consolidated Attendance Report
--------------------------------------------------


Start Date (YYYY-MM-DD) or press Enter for last 7 days:  
End Date (YYYY-MM-DD) or press Enter for today:  



Found 10 attendance records

✓ Consolidated Report Generated
..\Attendance\Consolidated_Attendance_2026-06-26_to_2026-07-03.xlsx


In [7]:
# List generated reports

attendance_dir = Path("../Attendance")

if attendance_dir.exists():

    print("\nGenerated Reports")
    print("-" * 40)

    for folder in attendance_dir.iterdir():

        if folder.is_dir():

            excel_files = list(folder.glob("*.xlsx"))

            if excel_files:

                print(f"\n{folder.name}/")

                for file in excel_files:

                    size = file.stat().st_size / 1024

                    print(
                        f"  {file.name} ({size:.1f} KB)"
                    )

else:

    print("Attendance directory not found.")


Generated Reports
----------------------------------------

2026-07-03/
  Attendance_ATT_20260703_112655_d25f21_2026-07-03.xlsx (5.7 KB)

department_reports/
  Computer Science_Attendance_2026-07-03.xlsx (5.6 KB)

student_reports/
  22001_Rahul Kumar_Attendance_2026-07-03.xlsx (5.4 KB)


In [8]:
# Close database connection

db.close()

print("\n✓ Database connection closed")

2026-07-03 11:29:32,037 - database - INFO - MongoDB connection closed



✓ Database connection closed
